In [3]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

data_path = Path("data/raw/dst_reviews_english_200.csv")

df = pd.read_csv(data_path)

print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

df.head()

Rows: 200
Columns: 21


,recommendation_id,language,review,voted_up,votes_up,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,written_during_early_access,timestamp_created,timestamp_updated,playtime_forever_minutes,playtime_at_review_minutes,num_games_owned,author_num_reviews,created_at,updated_at,playtime_forever_hours,playtime_at_review_hours
0,232114765,english,awesome game!,True,0,0,0.50000,0,True,False,False,1785934292,1785934292,1594,1564,268,52,2026-08-05 12:51:32+00:00,2026-08-05 12:51:32+00:00,26.566667,26.066667
1,232113302,english,The game is such an actual slog to play through that I genuinely do not understand how anyone could find it fun. \n\...,False,0,0,0.50000,0,True,False,False,1785932719,1785932719,164,164,610,12,2026-08-05 12:25:19+00:00,2026-08-05 12:25:19+00:00,2.733333,2.733333
2,232084852,english,its been 8 years and i havent reviewed this game?,True,0,0,0.50000,0,True,False,False,1785893162,1785893162,70614,70614,24,9,2026-08-05 01:26:02+00:00,2026-08-05 01:26:02+00:00,1176.900000,1176.900000
3,232074576,english,Very Nice,True,0,0,0.50000,0,True,False,False,1785880771,1785880771,576,576,0,1,2026-08-04 21:59:31+00:00,2026-08-04 21:59:31+00:00,9.600000,9.600000
4,232067811,english,I didn't love it and I never had a feeling 'today I would like to play DS' but when I was forced to play by friends ...,True,1,0,0.50641,0,False,False,False,1785874082,1785874082,4002,4002,223,18,2026-08-04 20:08:02+00:00,2026-08-04 20:08:02+00:00,66.700000,66.700000


In [4]:
df.columns.tolist()

['recommendation_id',
 'language',
 'review',
 'voted_up',
 'votes_up',
 'votes_funny',
 'weighted_vote_score',
 'comment_count',
 'steam_purchase',
 'received_for_free',
 'written_during_early_access',
 'timestamp_created',
 'timestamp_updated',
 'playtime_forever_minutes',
 'playtime_at_review_minutes',
 'num_games_owned',
 'author_num_reviews',
 'created_at',
 'updated_at',
 'playtime_forever_hours',
 'playtime_at_review_hours']

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 21 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   recommendation_id            200 non-null    int64  
 1   language                     200 non-null    object 
 2   review                       199 non-null    object 
 3   voted_up                     200 non-null    bool   
 4   votes_up                     200 non-null    int64  
 5   votes_funny                  200 non-null    int64  
 6   weighted_vote_score          200 non-null    float64
 7   comment_count                200 non-null    int64  
 8   steam_purchase               200 non-null    bool   
 9   received_for_free            200 non-null    bool   
 10  written_during_early_access  200 non-null    bool   
 11  timestamp_created            200 non-null    int64  
 12  timestamp_updated            200 non-null    int64  
 13  playtime_forever_min

In [6]:
missing_values = (
    df.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame(name="missing_count")
)

missing_values["missing_percentage"] = (
    missing_values["missing_count"] / len(df) * 100
).round(2)

missing_values

,missing_count,missing_percentage
review,1,0.5
recommendation_id,0,0.0
language,0,0.0
voted_up,0,0.0
votes_up,0,0.0
votes_funny,0,0.0
weighted_vote_score,0,0.0
comment_count,0,0.0
steam_purchase,0,0.0
received_for_free,0,0.0


In [7]:
print(
    "Duplicate review IDs:",
    df["recommendation_id"].duplicated().sum(),
)

print(
    "Duplicate review texts:",
    df["review"].duplicated().sum(),
)

Duplicate review IDs: 0
Duplicate review texts: 3


In [8]:
review_counts = (
    df["voted_up"]
    .value_counts(dropna=False)
    .rename(index={True: "Recommended", False: "Not recommended"})
)

review_counts

voted_up
Recommended        176
Not recommended     24
Name: count, dtype: int64

In [9]:
positive_rate = df["voted_up"].mean() * 100

print(f"Positive review rate: {positive_rate:.2f}%")

Positive review rate: 88.00%


In [10]:
df["created_at"] = pd.to_datetime(
    df["created_at"],
    utc=True,
    errors="coerce",
)

df["updated_at"] = pd.to_datetime(
    df["updated_at"],
    utc=True,
    errors="coerce",
)

print("Earliest review:", df["created_at"].min())
print("Latest review:", df["created_at"].max())

Earliest review: 2026-07-23 07:05:40+00:00
Latest review: 2026-08-05 12:51:32+00:00


In [11]:
df[
    [
        "playtime_at_review_hours",
        "playtime_forever_hours",
    ]
].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
)

,playtime_at_review_hours,playtime_forever_hours
count,200.000000,200.000000
mean,137.155917,144.592000
std,277.922911,281.003132
min,0.400000,0.400000
25%,7.666667,10.854167
50%,28.875000,36.975000
75%,111.216667,133.612500
90%,406.468333,413.483333
95%,709.702500,713.557500
99%,1227.889000,1264.028833


In [12]:
df["review_length_chars"] = (
    df["review"]
    .fillna("")
    .str.len()
)

df["review_word_count"] = (
    df["review"]
    .fillna("")
    .str.split()
    .str.len()
)

df[
    [
        "review_length_chars",
        "review_word_count",
    ]
].describe()

,review_length_chars,review_word_count
count,200.000000,200.000000
mean,88.600000,16.520000
std,144.788704,26.471177
min,0.000000,0.000000
25%,11.000000,3.000000
50%,40.500000,7.000000
75%,97.500000,18.000000
max,1103.000000,200.000000


In [13]:
df[
    [
        "review",
        "voted_up",
        "review_word_count",
    ]
].sort_values(
    "review_word_count",
    ascending=False,
).head()

,review,voted_up,review_word_count
138,"Every time I open my Steam library and see the game, I ask myself: I loved the single-player one, so why didn't I ke...",True,200
112,Most unique survival game of all time. No game by another development team will ever manage to recreate the magic th...,True,157
31,I love the art style of the game. Its a fun game if you take the time to learn the basics. It does have a steep lea...,True,155
10,В Don't Starve Together я играл несколько лет но в стиме купил недавно и хочу поделится о самой игре\nDon't Starve T...,True,88
1,The game is such an actual slog to play through that I genuinely do not understand how anyone could find it fun. \n\...,False,87


In [14]:
print(df.shape)
print(df["voted_up"].value_counts())
print(f"Positive rate: {df['voted_up'].mean() * 100:.2f}%")
print(df["created_at"].min())
print(df["created_at"].max())
print(df["recommendation_id"].duplicated().sum())

(200, 23)
voted_up
True     176
False     24
Name: count, dtype: int64
Positive rate: 88.00%
2026-07-23 07:05:40+00:00
2026-08-05 12:51:32+00:00
0
